# Homework 5 - Clustering

## EDA

In [ ]:
# Basic imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (8, 5)

# Sklearn imports for clustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score, v_measure_score,
    fowlkes_mallows_score
)




In [ ]:

def internal_metrics(X, labels, model_name):
    #filter out noise (-1) for DBSCAN
    mask = labels != -1 if -1 in labels else np.ones(len(labels), dtype=bool)

    return {
        "Model": model_name,
        "Silhouette": silhouette_score(X[mask], labels[mask]),
        "Calinski-Harabasz": calinski_harabasz_score(X[mask], labels[mask]),
        "Davies-Bouldin": davies_bouldin_score(X[mask], labels[mask]),
    }


def external_metrics(ref_labels, model_labels, model_name):
    # DBSCAN esetén -1 (noise) kiszűrése
    mask = model_labels != -1 if -1 in model_labels else np.ones(len(model_labels), dtype=bool)

    return {
        "Model": model_name,
        "ARI": adjusted_rand_score(ref_labels[mask], model_labels[mask]),
        "NMI": normalized_mutual_info_score(ref_labels[mask], model_labels[mask]),
        "V-measure": v_measure_score(ref_labels[mask], model_labels[mask]),
        "Fowlkes-Mallows": fowlkes_mallows_score(ref_labels[mask], model_labels[mask]),
    }



In [ ]:
df = pd.read_csv()

## 1.2 Univariate Analysis

I'll examine the descriptives and individual distributions of each numerical feature in the dataset.  

Because clustering relies on geometric distances in feature space, variables with high skewness or extreme outliers may dominate the clustering process if not properly transformed or scaled.


In [ ]:
# Work on a copy of the raw data
print("\nDescriptive statistics:")
display(df.describe())

# List of numerical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_cols)

# Plot histograms with KDE for each feature
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(
        df[col],
        kde=True,
        ax=axes[i],
        bins=30
    )
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()


# Boxplots to detect outliers
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(
        x=df[col],
        ax=axes[i]
    )
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_xlabel("")

plt.tight_layout()
plt.show()



## 1.3 Multivariate Analysis

I'll examine the relationships between all pairs of numerical features
using pairwise scatterplots and correlation analysis.

In [ ]:
# Pairwise scatterplots for all numerical features
sns.pairplot(
    df[numeric_cols],
    diag_kind="kde", #distributions are more continuous
    plot_kws={
        "alpha": 0.4,  #make overlaps visible
        "s": 20,
        "edgecolor": "none"
    },
    height=2.5
)

plt.suptitle(
    "Pairwise Relationships Between Earthquake Features",
    y=1.02,
    fontsize=14
)
plt.show()


# Correlation matrix
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()


## Data Preparation


In [ ]:
features = ["Focal depth", "Latitude", "Longitude", "Richter"]

X = df[features].copy()

### Transforming

Focal depth is very rightly skewed, so I try to log-transform it

In [ ]:
# Log-transform focal depth to reduce skewness
X_log = X.copy()
X_log["Focal depth"] = np.log1p(X_log["Focal depth"])

# Distribution of focal depth after log transformation
plt.figure(figsize=(6, 4))

sns.histplot(
    X_log["Focal depth"],
    kde=True,
    bins=30
)

plt.title("Focal depth after log transformation")
plt.xlabel("log(1 + Focal depth)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


### Scaling

Because clustering algorithms are sensitive to the scale of the input features,
standardization is applied to ensure that all variables contribute equally to
distance calculations. StandardScaler is appropriate given the continuous numerical nature of the variables and the absence of extreme heavy-tailed noise after transformation. RobustScaler could be an alternative, but log-transformation handled the focal depth outliers.


In [ ]:
# Standardize features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_log_scaled = scaler.fit_transform(X_log)

# Convert back to DataFrame for easier handling
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=df.index)
X_log_scaled_df = pd.DataFrame(X_log_scaled, columns=features, index=df.index)

## K-Means Clustering

### Baseline Model (k = 15)

In [ ]:
# Baseline K-Means with k=15 on scaled original features
kmeans_15 = KMeans(
    n_clusters=15,
    random_state=42,
    n_init=10
)
labels_15 = kmeans_15.fit_predict(X_scaled_df)
silhouette_15 = silhouette_score(X_scaled_df, labels_15)
print(f"Silhouette score (original scaled features, k=15): {silhouette_15:.4f}")


# Baseline K-Means with k=15 on log-transformed scaled features
kmeans_15_log = KMeans(
    n_clusters=15,
    random_state=42,
    n_init=10
)
labels_15_log = kmeans_15_log.fit_predict(X_log_scaled_df)
silhouette_15_log = silhouette_score(X_log_scaled_df, labels_15_log)

print(f"Silhouette score (log-transformed scaled features, k=15): {silhouette_15_log:.4f}")


### 3.2 Optimal Number of Clusters

I'll do the search for optiomal k for the original and the log-transformed scaled features to assess the impact of the transformation

I'll use elbow method and silhouette analysis


In [ ]:
# Range of cluster numbers to evaluate
k_range = range(2, 21)

#elbow for original
inertia_original = []

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled_df)
    inertia_original.append(kmeans.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(k_range, inertia_original, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method – Original Scaled Features")
plt.tight_layout()
plt.show()

#elbow for transformed
inertia_log = []

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_log_scaled_df)
    inertia_log.append(kmeans.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(k_range, inertia_log, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method – Log-Transformed Scaled Features")
plt.tight_layout()
plt.show()

#silhouette analysis for original
silhouette_original = []

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = kmeans.fit_predict(X_scaled_df)
    score = silhouette_score(X_scaled_df, labels)
    silhouette_original.append(score)

plt.figure(figsize=(6, 4))
plt.plot(k_range, silhouette_original, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Analysis – Original Scaled Features")
plt.tight_layout()
plt.show()

#silhouette analysis for transformed
silhouette_log = []

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = kmeans.fit_predict(X_log_scaled_df)
    score = silhouette_score(X_log_scaled_df, labels)
    silhouette_log.append(score)

plt.figure(figsize=(6, 4))
plt.plot(k_range, silhouette_log, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Analysis – Log-Transformed Scaled Features")
plt.tight_layout()
plt.show()




### 3.3 Cluster Visualization

A K-Means model with k = 6 clusters is selected for visualization.  
To further assess the impact of feature transformation, clusters are visualized for both the original and the log-transformed version.

Principal Component Analysis (PCA) is used to project the high-dimensional feature space into two dimensions for visualization purposes only.


In [ ]:
# K-Means with k=6 on original
kmeans_6 = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

labels_6 = kmeans_6.fit_predict(X_scaled_df)

# K-Means with k=6 on transformed
kmeans_6_log = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

labels_6_log = kmeans_6_log.fit_predict(X_log_scaled_df)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=df.index
)
pca_df["Cluster"] = labels_6

# PCA for visualization (log-transformed scaled features)
pca_log = PCA(n_components=2, random_state=42)
X_log_pca = pca_log.fit_transform(X_log_scaled_df)

pca_log_df = pd.DataFrame(
    X_log_pca,
    columns=["PC1", "PC2"],
    index=df.index
)
pca_log_df["Cluster"] = labels_6_log

#scatterplot original
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10",
    s=20,
    legend=True
)

plt.title("K-Means Clusters (k = 6) – Original Scaled Features")
plt.tight_layout()
plt.show()


#scatterplot transformed
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_log_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10",
    s=20,
    legend=True
)

plt.title("K-Means Clusters (k = 6) – Log-Transformed Scaled Features")
plt.tight_layout()
plt.show()



### Density-Based Spatial Clustering (DBSCAN)

In [ ]:
eps_values = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70]
min_samples_values = [3, 4, 5, 6, 7, 8, 9, 10, 12, 15]


#original dataset
dbscan_results_original = []

for eps in eps_values:
    for min_s in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_scaled_df)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        if n_clusters <= 1:
            continue

        silhouette = silhouette_score(X_scaled_df, labels)
        noise_ratio = np.mean(labels == -1)

        dbscan_results_original.append({
            "dataset": "Original scaled",
            "eps": eps,
            "min_samples": min_s,
            "n_clusters": n_clusters,
            "silhouette": silhouette,
            "noise_ratio": noise_ratio
        })


#log-transformed dataset
dbscan_results_log = []

for eps in eps_values:
    for min_s in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_log_scaled_df)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        if n_clusters <= 1:
            continue

        silhouette = silhouette_score(X_log_scaled_df, labels)
        noise_ratio = np.mean(labels == -1)

        dbscan_results_log.append({
            "dataset": "Log-transformed scaled",
            "eps": eps,
            "min_samples": min_s,
            "n_clusters": n_clusters,
            "silhouette": silhouette,
            "noise_ratio": noise_ratio
        })

# original dataset
dbscan_results_original_df = pd.DataFrame(dbscan_results_original)

top_original = (
    dbscan_results_original_df
    .sort_values("silhouette", ascending=False)
    .head(10)
)

display(top_original)


# log-transformed dataset
dbscan_results_log_df = pd.DataFrame(dbscan_results_log)

top_log = (
    dbscan_results_log_df
    .sort_values("silhouette", ascending=False)
    .head(10)
)

display(top_log)



In [ ]:
# Final DBSCAN configuration - Original scaled
dbscan_original_final = DBSCAN(eps=0.6, min_samples=15)
labels_db_original = dbscan_original_final.fit_predict(X_scaled_df)

# Final DBSCAN configuration - Log-transformed scaled
dbscan_log_final = DBSCAN(eps=0.7, min_samples=15)
labels_db_log = dbscan_log_final.fit_predict(X_log_scaled_df)

# Attach DBSCAN labels to existing PCA dataframes
pca_df["DBSCANCluster"] = labels_db_original
pca_log_df["DBSCANCluster"] = labels_db_log

#original
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="DBSCANCluster",
    palette="tab10",
    s=20,
    legend=True
)

plt.title("DBSCAN – Original Scaled Features")
plt.tight_layout()
plt.show()

# transformed
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_log_df,
    x="PC1",
    y="PC2",
    hue="DBSCANCluster",
    palette="tab10",
    s=20,
    legend=True
)

plt.title("DBSCAN – Log-Transformed Scaled Features")
plt.tight_layout()
plt.show()




In [ ]:

#Evaluating one-by-one
internal_results = []

# KMeans
internal_results.append(
    internal_metrics(X_scaled_df.values, labels_6, "KMeans – original scaled")
)
internal_results.append(
    internal_metrics(X_log_scaled_df.values, labels_6_log, "KMeans – log-transformed scaled")
)

#  DBSCAN
internal_results.append(
    internal_metrics(X_scaled_df.values, labels_db_original, "DBSCAN – original scaled")
)
internal_results.append(
    internal_metrics(X_log_scaled_df.values, labels_db_log, "DBSCAN – log-transformed scaled")
)

internal_df = pd.DataFrame(internal_results)
display(internal_df)


In [ ]:

reference_labels = labels_6  # KMeans (k=6, original scaled)

external_results = []

# DBSCAN
external_results.append(
    external_metrics(reference_labels, labels_db_original, "DBSCAN – original scaled")
)
external_results.append(
    external_metrics(reference_labels, labels_db_log, "DBSCAN – log-transformed scaled")
)

external_df = pd.DataFrame(external_results)
display(external_df)


###6.3 3D space

My next idea was to create a 3d plot,  longitude-latitude expanded with focal depth.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 18))

for i, (name, labels) in enumerate(models, start=1):
    ax = fig.add_subplot(4, 2, i, projection="3d")
    cm = build_color_map(labels)
    colors = labels_to_colors(labels, cm)

    ax.scatter(
        plot_df["Lon"], plot_df["Lat"], plot_df["Depth"],
        c=colors, s=10, alpha=0.65, depthshade=False
    )

    ax.set_title(name, fontsize=11, pad=8)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_zlabel("Focal depth")

    # Same camera angle for comparability
    ax.view_init(elev=18, azim=235)

fig.suptitle("6.3 Type 3 — 3D Lon–Lat–Depth (colored by cluster) for all 8 models", fontsize=15, y=0.995)
plt.tight_layout()
plt.show()
